# 🔬 Mercor Paper: Baseline vs. Upgraded GRPO on SWE-bench
## Reproducing "Training Frontier Knowledge Work Agents: A 397B RL Training Guide with SkyRL"
> **Paper**: [mercor.com/blog/training-frontier...](https://www.mercor.com/blog/training-frontier-knowledge-work-agents-a-397b-rl-training-guide-with-skyrl/)  
> **Model used here**: `Qwen/Qwen2.5-Coder-0.5B-Instruct` on SWE-bench Verified  
> **Expected runtime**: ~3 minutes on Colab T4 GPU  

### What this notebook does:
1. Runs **Baseline GRPO** (your original `swe_grpo_one_step.py`) and records time + results
2. Runs **Mercor-Upgraded GRPO** (with 3 surgical fixes from the paper) and records time + results  
3. Prints a **side-by-side comparison table** of both approaches

### ⚠️ Before running: Enable GPU
**Runtime → Change runtime type → T4 GPU → Save**, then **Runtime → Run all**

In [ ]:
# Install required packages
!pip install -q "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" "peft>=0.12" torch
# Colab ships torchao which conflicts with peft — remove it
!pip install -q --no-deps torchao 2>/dev/null || true

print("✅ Packages ready")

In [ ]:
import os
import json
import random
import time
import urllib.request

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

# Fetch the utility file from the original project repo
# (MockEnv, generate, first_bash_block, SYSTEM prompt, NO_COMMAND)
if not os.path.exists("utils.py"):
    print("Fetching utils.py from swe_in_prod_vizuara_01...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/abgoswam/swe_in_prod_vizuara_01/main/utils.py",
        "utils.py"
    )

from utils import NO_COMMAND, SYSTEM, MockEnv, first_bash_block, generate

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if device == "cuda" else "CPU only"
print(f"✅ Device: {device} | Hardware: {gpu_name}")

## Step 1: Load Config & SWE-bench Task
We pick the simplest task from SWE-bench Verified: one file changed, short patch.  
This is the exact same task used in your original `swe_grpo_one_step.py`.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
MODEL      = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
GROUP_SIZE = 4     # rollouts per task (use 4 to keep runtime short on free T4)
MAX_TURNS  = 3     # max agent turns per rollout
LR         = 1e-5  # AdamW learning rate
SEED       = 0
DPPO_DELTA = 0.2   # DPPO divergence threshold (Mercor Fix #3)

random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── LOAD SWE-BENCH TASK ──────────────────────────────────────────────────────
print("Loading SWE-bench Verified dataset (first download may take ~30s)...")
ds = load_dataset("princeton-nlp/SWE-bench_Verified", split="test")

# Filter: pick tasks with exactly 1 changed file and a short patch
cands = [i for i, r in enumerate(ds)
         if r["patch"].count("diff --git") == 1 and len(r["patch"]) < 1800]

inst         = ds[cands[0]]
fail_to_pass = json.loads(inst["FAIL_TO_PASS"])

print(f"\n✅ Task loaded!")
print(f"   Instance ID : {inst['instance_id']}")
print(f"   Repo        : {inst['repo']}")
print(f"   Tests (FAIL→PASS): {fail_to_pass}")
print(f"\nIssue preview:")
print(inst["problem_statement"][:400] + "...")

## Step 2: Shared Helpers (model loading, tokenizer, LoRA)
These functions are used by **both** the baseline and the Mercor version.

In [ ]:
def load_policy():
    """Load Qwen2.5-Coder-0.5B WITHOUT LoRA (for rollout collection)."""
    tok = AutoTokenizer.from_pretrained(MODEL)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        attn_implementation="sdpa"
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model loaded: {MODEL}")
    print(f"Parameters : {n_params:,} (none trainable yet — LoRA added later)")
    return model, tok


def add_lora(model):
    """Add rank-8 LoRA adapters to attention layers. Only these are trained."""
    model = get_peft_model(model, LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.0, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
    ))
    model.print_trainable_parameters()
    return model


def build_masked(messages, tokenizer, max_len=3072):
    """
    Tokenize the full conversation and set labels=-100 for non-assistant tokens.
    Only assistant tokens contribute to the loss.
    """
    ids, labels, prev = [], [], ""
    for i, m in enumerate(messages):
        cur = tokenizer.apply_chat_template(messages[:i + 1], tokenize=False)
        assert cur.startswith(prev), "chat template must be append-only"
        seg = tokenizer(cur[len(prev):], add_special_tokens=False)["input_ids"]
        ids    += seg
        labels += seg if m["role"] == "assistant" else [-100] * len(seg)
        prev = cur
    return ids[:max_len], labels[:max_len]


def reward_random(patch, rng):
    """Stand-in reward: 0 if no patch written, random float in [0,1] otherwise."""
    if not patch:
        return 0.0
    return round(float(rng.random()), 3)


print("✅ Helpers defined")

---
## 🔴 Part A: Baseline GRPO (Original `swe_grpo_one_step.py`)

This is your **original code** from [`swe_in_prod_vizuara_01`](https://github.com/abgoswam/swe_in_prod_vizuara_01).

### What the baseline does:
- Collects GROUP_SIZE rollouts of the agent trying to fix a bug
- Uses `token_mean` loss: averages across ALL tokens globally → **biased toward long rambling rollouts**
- No warning to the agent that it's running out of turns → **many rollouts end with reward = 0**
- No importance-ratio masking → **policy can drift and collapse**

In [ ]:
# ── BASELINE: Agent harness (no nudge) ──────────────────────────────────────
def run_agent_baseline(model, tok, inst, fail_to_pass):
    """
    Standard agent loop (ReAct style).
    
    ⚠️ BASELINE GAP: No nudge on the final turn.
    The model doesn't know time is running out → explores forever → reward = 0.
    """
    env     = MockEnv(fail_to_pass)
    context = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": f"ISSUE:\n{inst['problem_statement'][:1500]}"}
    ]

    for turn_idx in range(MAX_TURNS):
        prompt = tok.apply_chat_template(context, tokenize=False, add_generation_prompt=True)
        reply  = generate(model, tok, prompt, temperature=1.0)
        action = first_bash_block(reply)
        obs    = env.run(action) if action else NO_COMMAND
        context += [
            {"role": "assistant", "content": reply},
            {"role": "user",      "content": obs[:800]}
        ]
        # ⚠️ No nudge here — even on final turn

    return dict(messages=context, patch=env.patch(), calls=env.calls)


# ── BASELINE: token_mean log probability ─────────────────────────────────────
def seq_logprob_token_mean(model, tok, messages):
    """
    Baseline loss computation: average logprob across ALL tokens globally.
    
    ⚠️ PROBLEM: A 5000-token rollout contributes 25x more gradient signal
    than a 200-token rollout, regardless of which was better.
    """
    ids, labs = build_masked(messages, tok)
    t   = torch.tensor([ids], device=device)
    msk = torch.tensor([[0. if l == -100 else 1. for l in labs]], device=device)[:, 1:]
    logits = model(t).logits[:, :-1]
    lp = torch.log_softmax(logits.float(), -1).gather(-1, t[:, 1:].unsqueeze(-1)).squeeze(-1)
    # token_mean = sum / total_tokens (biased by sequence length!)
    return (lp * msk).sum() / msk.sum().clamp(min=1), int(msk.sum().item())


print("✅ Baseline functions defined")